# 4. 골든배치 선정 기준 확정

## 기준 설계

총수확량과 독립된 공정 안정성 조건을 먼저 적용한다. 현장 규격이 제공되지 않았으므로 정상 배치의 전략별 경험분포를 이용한 임시 게이트를 사용한다. 이 기준은 통계적 참고기준이며 장비·공정 허용한계를 대체하지 않는다. 확정 후보는 RC 8·12·17·26·14·16, OC 35·57·48, APC 79·85·65·62·82·68이다. CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


golden_ids = [8, 12, 17, 26, 14, 16, 35, 57, 48, 79, 85, 65, 62, 82, 68]
data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    late = time / time[-1] >= 0.8
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    substrate = batch['기질농도(g/L)'].to_numpy()
    rows.append({
        '배치번호': batch_number, '전략': 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC'),
        '선정': batch_number in golden_ids,
        '후기농도기울기': np.polyfit(time[late], penicillin[late], 1)[0],
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO최솟값': batch['용존산소(mg/L)'].min(),
        '기질최종': substrate[-1],
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '염기총투입량': np.trapezoid(batch['염기투입유량(L/h)'], time),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.groupby(['전략', '선정']).size().unstack())

선정,False,True
전략,,
APC,24,6
OC,27,3
RC,24,6


### 판단

확정 후보는 RC 6개, OC 3개, APC 6개다. 전략별 개수가 다르므로 전체 평균을 그대로 비교하지 않고 모든 기준을 전략별 분포에서 계산한다.

In [2]:
limit_rows = []
for strategy, group in batch_metrics.groupby('전략'):
    limit_rows.append({
        '전략': strategy,
        'pH상한_75분위': group['pH표준편차'].quantile(0.75),
        'DO하한_10분위': group['DO최솟값'].quantile(0.10),
        '기질상한_75분위': group['기질최종'].quantile(0.75),
        '산상한_90분위': group['산총투입량'].quantile(0.90),
        '염기상한_90분위': group['염기총투입량'].quantile(0.90),
    })
empirical_limits = pd.DataFrame(limit_rows)
display(empirical_limits.round(4))

evaluation = batch_metrics.merge(empirical_limits, on='전략')
evaluation['농도유지통과'] = evaluation['후기농도기울기'] >= 0
evaluation['pH통과'] = evaluation['pH표준편차'] <= evaluation['pH상한_75분위']
evaluation['DO통과'] = evaluation['DO최솟값'] >= evaluation['DO하한_10분위']
evaluation['기질통과'] = evaluation['기질최종'] <= evaluation['기질상한_75분위']
evaluation['산통과'] = evaluation['산총투입량'] <= evaluation['산상한_90분위']
evaluation['염기통과'] = evaluation['염기총투입량'] <= evaluation['염기상한_90분위']
check_columns = ['농도유지통과', 'pH통과', 'DO통과', '기질통과', '산통과', '염기통과']
evaluation['통과수'] = evaluation[check_columns].sum(axis=1)
evaluation['전체통과'] = evaluation[check_columns].all(axis=1)
selected_evaluation = evaluation.loc[evaluation['선정'], [
    '배치번호', '전략', *check_columns, '통과수', '전체통과',
    '후기농도기울기', 'pH표준편차', 'DO최솟값', '기질최종', '산총투입량', '염기총투입량',
]].sort_values(['전략', '통과수', '배치번호'])
display(selected_evaluation.round(4))
display(selected_evaluation.groupby('전략')['전체통과'].agg(['sum', 'count']).rename(columns={'sum': '전체통과수', 'count': '선정수'}))

,전략,pH상한_75분위,DO하한_10분위,기질상한_75분위,산상한_90분위,염기상한_90분위
0,APC,0.0261,4.1534,0.0016,4.4752,22046.1556
1,OC,0.0305,5.8239,40.2890,44.1889,18915.0209
2,RC,0.0290,6.0710,42.6935,42.5901,16873.2715


,배치번호,전략,농도유지통과,pH통과,DO통과,기질통과,산통과,염기통과,통과수,전체통과,후기농도기울기,pH표준편차,DO최솟값,기질최종,산총투입량,염기총투입량
67,68,APC,True,False,False,True,False,True,3,False,0.0768,0.0284,2.6932,0.0015,7.2663,17893.6613
78,79,APC,True,True,True,False,True,True,5,False,0.0472,0.0183,8.8312,0.0021,1.6616,14728.7789
61,62,APC,True,True,True,True,True,True,6,True,0.0730,0.0183,8.1236,0.0014,2.4930,12623.0148
64,65,APC,True,True,True,True,True,True,6,True,0.0721,0.0245,7.9215,0.0015,1.6614,18799.4642
81,82,APC,True,True,True,True,True,True,6,True,0.0630,0.0164,9.6106,0.0014,1.6619,15270.6219
84,85,APC,True,True,True,True,True,True,6,True,0.1381,0.0162,9.6297,0.0015,1.6627,10711.7686
47,48,OC,True,True,True,True,True,False,5,False,0.0687,0.0209,9.5810,1.0812,5.3500,19360.8578
34,35,OC,True,True,True,True,True,True,6,True,0.1151,0.0240,8.5707,0.0017,4.7758,15888.1245
56,57,OC,True,True,True,True,True,True,6,True,0.0519,0.0203,10.1850,0.0018,2.9130,15723.0645
16,17,RC,True,True,True,True,True,False,5,False,0.1326,0.0287,8.0054,0.0013,5.9156,18639.6422


,전체통과수,선정수
전략,,
APC,4,6
OC,2,3
RC,4,6


### 판단

15개 중 10개가 여섯 가지 경험적 게이트를 모두 통과한다. RC 17·26과 OC 48은 염기 총투입량 90분위 기준만 초과한다. APC 79는 기질 기준만 초과하지만 실제 최종 기질은 0.0021g/L로 매우 작아 통계적 분위수와 실무 중요성을 구분해야 한다. APC 68은 pH·DO·산 투입의 세 기준을 동시에 위반해 가장 강한 재검토 대상이다.

In [3]:
stability_metrics = ['pH표준편차', 'DO최솟값', '기질최종', '산총투입량', '염기총투입량']
standardized = batch_metrics.copy()
for metric in stability_metrics:
    standardized[metric] = standardized.groupby('전략')[metric].transform(lambda x: (x - x.mean()) / x.std(ddof=1))
direction = {'pH표준편차': -1, 'DO최솟값': 1, '기질최종': -1, '산총투입량': -1, '염기총투입량': -1}
standardized['안정성점수'] = sum(direction[metric] * standardized[metric] for metric in stability_metrics) / len(stability_metrics)
summary = standardized.groupby(['전략', '선정'])['안정성점수'].agg(['mean', 'median', 'count']).unstack()
display(summary.round(4))

HOLDOUT_BATCH_IDS = []
if not HOLDOUT_BATCH_IDS:
    print('독립 홀드아웃 배치가 없어 최종 외부 검증은 수행할 수 없습니다.')


mean          median         count      
선정    False   True    False   True  False True 
전략                                             
APC -0.0136  0.0543  0.0212  0.2245    24     6
OC  -0.0397  0.3571  0.0498  0.3301    27     3
RC  -0.0868  0.3472  0.0131  0.3595    24     6

독립 홀드아웃 배치가 없어 최종 외부 검증은 수행할 수 없습니다.


### 최종 판단과 결론

- 현재 선정은 고성과 후보군으로는 합당하지만 공정 안정성 게이트까지 적용하면 10/15개만 전부 통과한다.
- 17·26·48은 염기 과다 여부를 도메인 허용범위로 재확인하고, 79는 APC의 매우 좁은 기질 분포 때문에 발생한 통계적 실패인지 확인한다.
- 68은 DO 최솟값 2.6932mg/L, pH 표준편차 0.0284, 산 총투입량 7.2663으로 APC 내 여러 안정성 기준을 동시에 위반한다. 골든배치 확정 전 궤적 검토 또는 조건부 제외가 필요하다.
- 경험적 분위수는 현장 규격이 아니다. 최종 승인에는 장비·공정 허용범위와 독립 홀드아웃 배치가 필요하며, 현재 데이터에는 홀드아웃이 없어 최종 외부 검증은 미완료다.